<a href="https://colab.research.google.com/github/Realbenjizo/python-functions-for-data-engineers/blob/main/Chapter_13_Testing_Pipeline_Functions_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import logging
from typing import Dict

In [3]:
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

## **Unit Testing Pure Funtions**

In [5]:
import pandas as pd

df = pd.DataFrame([
    {'transaction_id': 1, 'user_id': 'u123', 'amount': 100, 'currency': 'USD'},
    {'transaction_id': 2, 'user_id': 'u456', 'amount': 50, 'currency': 'EUR'},
])
# Dictionaries of rates and tiers
rates = {'USD': 1.0, 'EUR': 1.09}
tiers = {'u123': 'gold', 'u456': 'basic'}
cashback_rates = {'gold': 0.05, 'silver': 0.02, 'basic': 0.01}

In [6]:
from typing import Dict

def enrich_transformations(
        df: pd.DataFrame,
        rates: Dict[str, float],
        tiers: Dict[str, str],
        cashback_rates: Dict[str, float]
    ) -> pd.DataFrame:

    """Pure: Same inputs; same outputs. No I/O, no side effects."""

    results = df.copy()

    results['exchange_rate'] = results['currency'].map(rates).fillna(1.0)
    results['user_tier'] = results['user_id'].map(tiers)

    results['amount_usd'] = results['amount'] * results['exchange_rate']
    results['cash_back'] = results['amount'] * results['user_tier']\
                                        .map(cashback_rates).fillna(0)
    return results[["user_id", "user_tier", "amount_usd", "cash_back"]]

if __name__ == "__main__":
    print(enrich_transformations(df, rates, tiers, cashback_rates))

  user_id user_tier  amount_usd  cash_back
0    u123      gold       100.0        5.0
1    u456     basic        54.5        0.5


## **Property Tests**

In [7]:
import logging

logger = logging.getLogger(__name__)

def test_enrich_transformations_properties(df):

    # Data
    rates = {'USD': 1.0, 'EUR': 1.09, 'GBP': 1.27}
    tiers = {'u123': 'gold', 'u456': 'basic'}
    cashback_rates = {'gold': 0.05, 'basic': 0.01}

    # Transformation step
    result = enrich_transformations(df, rates, tiers, cashback_rates)

    # Assert (Property tests)
    assert len(result) == len(df), "Row count was not preserved"
    assert 'amount_usd' in result.columns, "Required column 'amount_usd' is missing"
    assert result['amount_usd'].notna().all(), "Output contains unexpected null values"
    assert pd.api.types.is_numeric_dtype(result['amount_usd']), "'amount_usd' must be numeric"
    assert result['user_tier'].isin(['gold', 'basic']).all(), "Invalid user tier values detected"

    expected_columns = { "user_id", "user_tier", "amount_usd","cash_back"}
    assert set(result.columns) == expected_columns, "Final DataFrame schema mismatch"


if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(message)s")
    test_enrich_transformations_properties(df)
    logger.info("All property assertions passed")

All property assertions passed


## **Aggregate Tests**

In [8]:
import numpy as np

def test_enrich_transformations_aggregates(df):

    rates = {'USD': 1.0, 'EUR': 1.09, 'GBP': 1.27}
    tiers = {'u123': 'gold', 'u456': 'basic'}
    cashback_rates = {'gold': 0.05, 'basic': 0.01}

    # Execute the transformation
    result = enrich_transformations(df, rates, tiers, cashback_rates)

    # Isolate the input rows where currency is already USD
    usd_inputs = df[df['currency'] == 'USD']

    if len(usd_inputs) > 0:
        # Extract the corresponding output rows using the matching index
        usd_outputs = result.loc[usd_inputs.index]

        # Verify the USD amount remains unchanged
        amounts_match = np.isclose(usd_outputs['amount_usd'], usd_inputs['amount'])
        assert amounts_match.all(), "USD conversion mismatch"


if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(message)s")
    test_enrich_transformations_aggregates(df)
    logger.info("Aggregate business rules validated successfully!")

Aggregate business rules validated successfully!


# **Testing Functions with Side Effects**

In [9]:
import requests
import logging
from typing import Dict, Optional

logger = logging.getLogger(__name__)

def fetch_user_from_api(user_id: int) -> Optional[Dict]:
    """Fetch a single user, returning None gracefully if the request fails."""
    url = f"https://jsonplaceholder.typicode.com/users/{user_id}"

    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        return response.json()
    except (requests.exceptions.HTTPError, requests.exceptions.RequestException) as e:
        # Catch network/HTTP issues specifically.
        logger.warning(f"Network error fetching user {user_id}: {e}")
        return None

if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

    user_data = fetch_user_from_api(4)
    if user_data is not None:
        print(f"Successfully found user: {user_data.get('name')}")
    else:
        logger.error("Error occurred: User data is missing. Aborting operation.")

Successfully found user: Patricia Lebsack


# **Test the success**

In [10]:
import logging
from unittest.mock import patch, Mock

logger = logging.getLogger(__name__)

def test_fetch_user_from_api_success():
    """Test that a successful API call accurately returns the user data."""

    # Set up the Moke response object
    mock_response = Mock()
    mock_response.json.return_value = {"id": 4, "name": "Patricia Lebsack"}
    mock_response.raise_for_status.return_value = None  # Simulate an HTTP 200 Success

    # Intercept requests.get and force it to return our fake response
    with patch("requests.get", return_value=mock_response) as mock_get:
        result = fetch_user_from_api(4)

        # Check that we called the right URL and timeout rules
        mock_get.assert_called_once_with("https://jsonplaceholder.typicode.com/users/4", timeout=5)

        # Make sure the function returned the data exactly as expected
        assert result is not None, "The function unexpectedly returned None"
        assert result["id"] == 4, "The user ID does not match"
        assert result["name"] == "Patricia Lebsack", "The username does not match"


if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
    test_fetch_user_from_api_success()
    logger.info("Success path mock test passed perfectly!")

Success path mock test passed perfectly!


## **Testing the Failure Path**

In [11]:
import logging
from unittest.mock import patch, Mock
import requests

logger = logging.getLogger(__name__)

def test_fetch_user_from_api_http_error():
    """Test that an HTTP error results in a graceful None fallback."""

    mock_response = Mock()

    # Force raise_for_status to trigger a real HTTPError
    mock_response.raise_for_status.side_effect = requests.exceptions.HTTPError("404 Client Error")

    # Intercept requests.get and return our error-triggering response
    with patch("requests.get", return_value=mock_response):
        result = fetch_user_from_api(999)  # User 999 does not exist

        # Check that code caught the HTTP error and returned None
        assert result is None, "Should have returned None when an HTTP error occurs!"

if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
    test_fetch_user_from_api_http_error()
    logger.info("HTTP Error path mock test passed!")

Network error fetching user 999: 404 Client Error
HTTP Error path mock test passed!


# **Property-Based Testing (Advanced)**

In [17]:
%pip install hypothesis

In [13]:
from hypothesis import given, strategies as st, settings
import pandas as pd

# Define schema strategies
@given(
    st.lists(
        st.tuples(
            st.sampled_from(['u123', 'u456']),   # user_id
            st.floats(min_value=1, max_value=1000,
                      allow_nan=False, allow_infinity=False),
            st.sampled_from(['USD', 'EUR', 'GBP'])             # currency
        ),
        min_size=1,
        max_size=50
    )
)
@settings(max_examples=100) # Hypothesis to run 100 random datasets
def test_enrich_transformations_generative(data):

    df = pd.DataFrame(data, columns=["user_id", "amount", "currency"])

    rates = {'USD': 1.0, 'EUR': 1.09, 'GBP': 1.27}
    tiers = {'u123': 'gold', 'u456': 'basic'}
    cashback_rates = {'gold': 0.05, 'basic': 0.01}

    # Execute the transformation
    result = enrich_transformations(df, rates, tiers, cashback_rates)

    # 3. Assert our invariant structural properties hold true across all 100 runs
    assert len(result) == len(df), "Row count changed during transformation"
    assert 'amount_usd' in result.columns, "Schema missing 'amount_usd'"
    assert result['amount_usd'].notna().all(), "Pipeline allowed null values"
    assert pd.api.types.is_numeric_dtype(result['amount_usd']), "Data type is not numeric"
    assert result['user_tier'].isin(['gold', 'basic']).all(), "Invalid tier leaked"

    # Validate final structural contract schema matches exactly
    output_columns = {"user_id", "user_tier", "amount_usd", "cash_back"}
    assert set(result.columns) == output_columns, "DataFrame output schema mismatch"

# Execute function
test_enrich_transformations_generative()

In [14]:
if __name__ == "__main__":
    logger.info("Initializing Hypothesis ...")

    # Calling the function
    test_enrich_transformations_generative()

    logger.info("Hypothesis generated 100 distinct datasets. All property tests held true")

Initializing Hypothesis ...
Hypothesis generated 100 distinct datasets. All property tests held true


# **Challenge**

In [15]:
import pandas as pd

# Transaction data
transactions_df = pd.DataFrame({
    'transaction_id': [1001, 1002],
    'category': ['  Electronics ', 'FOOD'],
    'amount': [1250.50, 45.00]
})

def sanitize_transactions(df: pd.DataFrame) -> pd.DataFrame:
    """Sanitizes transaction data and flags high-risk transactions."""
    processed = df.copy()

    # Normalize category text
    processed['category'] = processed['category'].str.lower().str.strip()
    # Flag high-risk transactions
    processed['is_high_risk'] = processed['amount'] > 500.0

    return processed[['transaction_id', 'category', 'amount', 'is_high_risk']]

print(sanitize_transactions(transactions_df))

   transaction_id     category  amount  is_high_risk
0            1001  electronics  1250.5          True
1            1002         food    45.0         False


# **Solution**

In [16]:
import logging
import pandas as pd

logger = logging.getLogger(__name__)

def test_properties(df: pd.DataFrame) -> None:
    """Test the structural properties of the transformed DataFrame."""
    results = sanitize_transactions(df)

    assert len(results) == len(df), "Row count was not preserved"
    assert list(results.columns) == [
        "transaction_id",
        "category",
        "amount",
        "is_high_risk"
    ], "Output columns do not match the expected schema"
    assert pd.api.types.is_bool_dtype(
        results["is_high_risk"]), "'is_high_risk' is not a boolean type"


def test_high_risk_boundary() -> None:
    """Test boundary logic dynamically using vectorized masks."""

    # A boundary test case DataFrame
    boundary_df = pd.DataFrame({
        'transaction_id':[101, 102, 103],
        'category': ['test', 'test', 'test'],
        'amount': [499.99, 500.00, 500.01]  # Testing the exact edge values
    })

    results = sanitize_transactions(boundary_df)

    # Condition A: Amounts less than or equal to $500 must be False
    normal_risk_mask = results['amount'] <= 500.00
    normal_results = results.loc[normal_risk_mask]
    assert (~normal_results['is_high_risk']).all(), "Amount <= 500 flagged as high risk"

    # Condition B: Amounts strictly over $500 must be True
    high_risk_mask = results['amount'] > 500.00
    high_results = results.loc[high_risk_mask]
    assert high_results['is_high_risk'].all(), "Amount > 500 failed to flag as high risk"

# Verification Code Block
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format="%(message)s")
    # Testing properties
    test_properties(transactions_df)
    # Testing high risk rules
    test_high_risk_boundary()
    logger.info("All property and dynamic boundary tests validated successfully!")

All property and dynamic boundary tests validated successfully!
